# Hybrid Gait Day 13 — 網格穩定性與事實輸入

**這份 notebook 記錄 Day 13 第一輪的量測結果。**
規劃在 `day13/day13_plan_zh_TW.md`，逐步紀錄在
`day13/day13_implementation_log_zh_TW.md`（新對話請先讀那一份）。

Day 13 的範圍是「**不需要 ABAD 就能解的部分**」——
越障五項失敗裡只有 `support_margin` 一項是 ABAD 的事，
而且 ABAD 應該最後做（它是在既定的支撐幾何上做最佳化）。

這一輪做了兩件事：

1. **C1–C3 三個事實輸入**問到答案（C3 結案）
2. **D1「滾動天花板」的前提查證**——結論是那個前提**不成立**

> **本輪最重要的結果不是一個數字，是一個否定：**
> 「142–158 mm 掃出天花板」這個工作項**現在不該做**，
> 因為可行性判準本身對搜尋網格敏感。


---

## 0. 環境


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name != "LegWheel" and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT.parent) not in sys.path:
    sys.path.insert(0, str(ROOT.parent))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib
# Day 12 log 陷阱 11：nbclient 必須明講 inline backend，
# 否則每張圖都只會變成 "Agg is non-interactive" 警告，不產生圖。
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 46)

NOTES = ROOT / "hybrid_note" / "notes"
DAY13 = NOTES / "day13"
DAY6_7 = NOTES / "day6-7"
print("LegWheel  ", ROOT)
print("day13 out ", DAY13)

LegWheel   /home/chang/corgi_ws/icra hybrid/LegWheel
day13 out  /home/chang/corgi_ws/icra hybrid/LegWheel/hybrid_note/notes/day13


---

## 1. C1–C3：三個事實輸入，repo 裡推導不出來

Day 13 規劃把 C 類排在最前面，理由是「最便宜，而且會改變你怎麼讀
**現有所有** margin 數字」。

**但先要確認的是：這三項我能不能自己量。** 查證結果是不能——
它們是**事實輸入**，跟「重心在幾何中心」同一類。
`legwheel` 沒有質量、沒有慣量、沒有勁度，也沒有 URDF 的 `<mass>` 標籤。


In [2]:
facts = pd.DataFrame([
    ("C3", "330 rpm 是關節側還是馬達側？",
     "關節側，無減速比", "結案",
     "1980 deg/s 就是關節速度，平地峰值 95.0% 的判讀【不變】"),
    ("C1", "腿的等效剛度（抬腳時支撐腿下沉多少）",
     "沒量過", "仍為缺口",
     "下沉量無法估；擋它的只有靜態 margin >= 3 mm floor"),
    ("C2", "重心偏移 ±2 mm/軸 是哪來的",
     "保守猜測，不是量測", "仍為缺口",
     "margin floor 3 mm 【建立在一個猜測上】"),
], columns=["項", "問題", "擁有者回答", "狀態", "後果"])
facts

,項,問題,擁有者回答,狀態,後果
0,C3,330 rpm 是關節側還是馬達側？,關節側，無減速比,結案,1980 deg/s 就是關節速度，平地峰值 95.0% 的判讀【不變】
1,C1,腿的等效剛度（抬腳時支撐腿下沉多少）,沒量過,仍為缺口,下沉量無法估；擋它的只有靜態 margin >= 3 mm floor
2,C2,重心偏移 ±2 mm/軸 是哪來的,保守猜測，不是量測,仍為缺口,margin floor 3 mm 【建立在一個猜測上】


### 1.1 這三個答案合起來，改變了 margin 該怎麼被引用

C3 是好消息（假設本來就對），C1/C2 則把 margin 的**成色**往下修。
平地那個 4.839 mm 要這樣拆開讀：

```text
 4.839 mm   量出來的（幾何，硬）
-3.000 mm   floor：其中 2 mm 是【猜的】，靈敏度 1.412 mm/mm 是量的
=1.839 mm   留給下沉（沒量）、柔度（沒量）、動力學（沒建模）、地形誤差
```

**只有 4.839 這個數字本身是硬的；它要對抗的門檻、以及剩下的餘裕，兩邊都不硬。**
論文與上機都不應該只寫「4.839 > 3，STABLE」。


---

## 2. D1：先查前提，再決定要不要掃

D1 原本寫的是「掃 142–158 mm，夾緊滾動天花板」。
現況只知道 `>140 mm`、`<=160 mm`。

**但在花八分鐘一格去掃之前，先看現有那 70 格的失敗長什麼樣。**
這是 Day 13 規劃 §6 的規則：
*一個「不可行」的結論，要先排除是自己問錯。*


In [3]:
sweep = pd.read_csv(DAY6_7 / "day6_7_step11r_feasibility_sweep.csv")
grid = (sweep.assign(h_mm=(sweep["obstacle_height_m"] * 1e3).round(0).astype(int),
                     th=sweep["theta_climb_deg"].round(0).astype(int))
             .pivot(index="h_mm", columns="th", values="feasible")
             .replace({True: "✓", False: "·"}))
print("已發表的 70 格滾動可行性（arc_samples=121, beta_step=1 度）")
grid

已發表的 70 格滾動可行性（arc_samples=121, beta_step=1 度）


th,40,45,50,55,60,65,70,75,80,85
h_mm,,,,,,,,,,
40,✓,✓,✓,✓,·,✓,✓,·,·,·
60,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓
80,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓
100,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓
120,·,✓,·,✓,·,·,·,·,·,·
140,✓,✓,·,✓,·,·,✓,·,·,·
160,·,·,·,·,·,·,·,·,·,·


**看 h = 140 那一列：`45 ✓  50 ·  55 ✓  60 ·  70 ✓`。**

真正的幾何天花板**不會在 theta 上交替**。交替是「解析度」的特徵，
不是「極限」的特徵。這就是要去查的線索。

失敗原因幾乎全是 `COUPLED_RESET_COLLISION_BLOCKED`。
讀 `single_leg_rolling_scene_2d.py` 那段：它是一個**以步長為格點的離散局部搜尋**，
所有候選都不合法時才報這個——也就是
「**在這組候選點裡找不到**」，不是「不存在」。


In [4]:
print(sweep.loc[sweep["obstacle_height_m"].between(0.139, 0.141),
                ["theta_climb_deg", "feasible", "failure_stage", "failure_reason"]]
            .to_string(index=False))

 theta_climb_deg  feasible            failure_stage                  failure_reason
            40.0      True                      NaN                             NaN
            45.0      True                      NaN                             NaN
            50.0     False LEFT_RIM_TRANSITION_FAIL COUPLED_RESET_COLLISION_BLOCKED
            55.0      True                      NaN                             NaN
            60.0     False LEFT_RIM_TRANSITION_FAIL COUPLED_RESET_COLLISION_BLOCKED
            65.0     False LEFT_RIM_TRANSITION_FAIL COUPLED_RESET_COLLISION_BLOCKED
            70.0      True                      NaN                             NaN
            75.0     False LEFT_RIM_TRANSITION_FAIL COUPLED_RESET_COLLISION_BLOCKED
            80.0     False LEFT_RIM_TRANSITION_FAIL COUPLED_RESET_COLLISION_BLOCKED
            85.0     False LEFT_RIM_TRANSITION_FAIL COUPLED_RESET_COLLISION_BLOCKED


---

## 3. 收斂測試：把搜尋步長減半，看判定往哪裡收斂

方法照 Day 13 log §11.4 對馬達速率那次：**加密網格，看數字往哪裡收斂**，
不要相信任何單一網格上的值。

### 3.1 過程中踩到的兩個量測錯誤（都在下結論之前抓到）

**錯誤一：細化步長會餓死步數預算。**
那些上限全部是**次數**不是距離（`roll_up_max_forward_steps` 100、
`approach_max_steps` 200、frame 上限 800/1200/500）。
步長減半，同一段實體距離就要走兩倍步數，預算卻沒加——
於是九格裡七格報 `MAX_FORWARD_STEPS_BEFORE_TOP_ROLL_COMPLETE`。

> 那個訊息讀起來像「滾不完」，其實是**我自己把預算調到不夠**。
> 特別危險：若沒注意失敗**原因換了種類**，只看「0.5 度全部不可行」，
> 會得出「加密後更不可行，所以 140 mm 天花板是真的」——**一個反向的錯誤結論**。

修正：步長縮的同時，預算以 `grow = 1/factor` 同步放大。

**錯誤二：只記 verdict 無法辨識「有東西消失」。**
如果某個 stage 悄悄被截斷，我只會看到「可行格數變了」，
分不出是真的改變還是有東西不見了——正是 body conflict 16→0 的同一個陷阱。

修正：加記 `frame_count`、`phases_visited`、五個 stage 各自的成敗。


In [5]:
sens = pd.read_csv(DAY13 / "day13_d1_grid_sensitivity.csv")
print("格數:", len(sens),
      "| 例外:", int((sens["stage"] == "EXCEPTION").sum()),
      "| 被預算餓死:", int(sens["reason"].str.contains("MAX_FORWARD_STEPS").sum()))

# 「有沒有東西消失」的檢查：細網格必須做【更多】工，不是更少
wide = sens.pivot_table(index=["height_mm", "theta_deg"], columns="factor",
                        values="frame_count")
print("\nframe_count 在 0.5 度變多的格數:", int((wide[0.5] > wide[1.0]).sum()), "/", len(wide))

格數: 60 | 例外: 0 | 被預算餓死: 0

frame_count 在 0.5 度變多的格數: 24 / 30


`例外 0`、`被預算餓死 0`，而且 frame_count 在多數格子**變多**——
**細網格是做了更多工，不是更少。**
所以接下來看到的差異不是「有東西被截斷」，是真的判定不同。


In [6]:
def verdict(r):
    return "FEAS" if r["feasible"] else (str(r["reason"]).split(":")[0][:22] or "-")

tbl = (sens.assign(v=sens.apply(verdict, axis=1))
           .pivot_table(index=["height_mm", "theta_deg"], columns="factor",
                        values="v", aggfunc="first"))
tbl.columns = ["1.0度", "0.5度"]
tbl["一致?"] = np.where(tbl["1.0度"] == tbl["0.5度"], "", "<<< 翻面")
tbl

1.0度                    0.5度     一致?
height_mm theta_deg                                                        
120.0     40.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          45.0       COUPLED_RESET_COLLISIO                    FEAS  <<< 翻面
          50.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          55.0       COUPLED_RESET_COLLISIO                    FEAS  <<< 翻面
          60.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          65.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          70.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          75.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          80.0       COUPLED_RESET_COLLISIO  NO_SLIP_REQUIRES_NEGAT  <<< 翻面
          85.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
140.0     40.0       COUPLED_RESET_COLLISIO                    FEAS  <<< 翻面
          45.0                         FEAS                    FEAS        
          50.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          55.0       COUPLED_RESET_COLLISIO                    FEAS  <<< 翻面
          60.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          65.0                         FEAS  COUPLED_RESET_COLLISIO  <<< 翻面
          70.0                         FEAS                    FEAS        
          75.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          80.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          85.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
160.0     40.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          45.0       COUPLED_RESET_COLLISIO  NO_SLIP_REQUIRES_NEGAT  <<< 翻面
          50.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          55.0       COUPLED_RESET_COLLISIO  NO_LEGAL_CORNER_PIVOT_  <<< 翻面
          60.0       NO_LEGAL_CORNER_PIVOT_  NO_LEGAL_CORNER_PIVOT_        
          65.0       COUPLED_RESET_COLLISIO  NO_LEGAL_CORNER_PIVOT_  <<< 翻面
          70.0       NO_LEGAL_CORNER_PIVOT_  COUPLED_RESET_COLLISIO  <<< 翻面
          75.0       NO_LEGAL_CORNER_PIVOT_  NO_LEGAL_CORNER_PIVOT_        
          80.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          85.0       NO_LEGAL_CORNER_PIVOT_  NO_LEGAL_CORNER_PIVOT_

In [7]:
both = int(((sens.pivot_table(index=["height_mm","theta_deg"], columns="factor",
                              values="feasible", aggfunc="first")).sum(axis=1) == 2).sum())
f10 = sens[(sens.factor == 1.0) & sens.feasible][["height_mm","theta_deg"]].apply(tuple, axis=1)
f05 = sens[(sens.factor == 0.5) & sens.feasible][["height_mm","theta_deg"]].apply(tuple, axis=1)
only10, only05 = set(f10) - set(f05), set(f05) - set(f10)

print(f"兩個網格都可行 : {both} / 30")
print(f"只有 1.0 度可行 : {len(only10)}   {sorted(only10)}")
print(f"只有 0.5 度可行 : {len(only05)}   {sorted(only05)}")
print(f"判定不一致      : {len(only10) + len(only05)} / 30 = "
      f"{(len(only10)+len(only05))/30*100:.0f}%")

兩個網格都可行 : 2 / 30
只有 1.0 度可行 : 4   [(120.0, 45.0), (120.0, 55.0), (140.0, 40.0), (140.0, 55.0)]
只有 0.5 度可行 : 1   [(140.0, 65.0)]
判定不一致      : 5 / 30 = 17%


### 3.2 關鍵觀察：判定不是收斂，是【雙向洗牌】

```text
h=140 th=40   可行  -> 不可行
h=140 th=55   可行  -> 不可行
h=140 th=65  不可行 ->  可行      <- 反方向
h=120 th=45   可行  -> 不可行
h=120 th=55   可行  -> 不可行
```

**如果細化只是「更嚴格」，改變會是單向的。**
`h=140 th=65` 由不可行變可行，證明不是變嚴格，**是換了搜尋路徑**。


---

## 4. 機制：為什麼不會收斂

追到 `single_leg_rolling_scene_2d.py:2262-2276`：

```python
desired_beta = previous_beta + top_beta_step
beta_values  = [desired_beta]
beta_search_count = floor(beta_search_window_rad / beta_search_step_rad)
for index in range(1, beta_search_count + 1):
    beta_values.append(desired_beta + index * top_beta_step)   # <- 關鍵
```

**候選解是以 `top_beta_step` 的整數倍產生的**，
而收斂測試縮的正是 `roll_up_top_beta_step_rad`。

```text
以為在做   同一個問題、網格更細  -> 答案應該收斂
實際在做   換掉整組候選解        -> 答案沒有理由收斂
```

步長同時扮演兩個角色：**路徑解析度** ＋ **候選解產生器**。
這兩個角色本來就該分開。

所以 `COUPLED_RESET_COLLISION_BLOCKED` 不是幾何結論，
它是「**這一組候選點裡沒有合法的**」。


---

## 5. 乾淨的後續實驗：只放寬搜尋範圍，路徑不動

§4 的機制說：步長同時扮演**路徑解析度**和**候選解產生器**兩個角色。
要把兩者分開，就要找一個**只動候選數量、不動路徑**的旋鈕。

`beta_search_window_rad` 正是那個旋鈕（`single_leg_rolling_scene_2d.py:4170`）：

```python
search_count = floor(beta_search_window / beta_step)
search_beta  = previous_beta + signed_step * beta_step
```

```text
beta_step     候選點的【間距】 —— 同時是路徑解析度
window        搜尋伸多遠的【範圍】 —— 只透過 search_count 決定候選【數量】
```

**固定 step、只放寬 window ＝ 在同樣的間距上多給候選點。**

> **這需要動專案程式碼**（`wheel_beta_search_window_rad` 原本沒接到
> `TraversalConstraints2D`，一直吃預設 5 度），**已取得專案擁有者同意**。
> 改動是純加法、預設值就是現有掃描實際跑的 5 度；
> 已驗證預設行為逐位元不變，且 `test_right_up_left_down_*` 三檔 **68 passed**。


In [8]:
win = pd.read_csv(DAY13 / "day13_d1_search_window.csv")
print("格數:", len(win),
      "| 例外:", int((win["stage"] == "EXCEPTION").sum()),
      "| 被預算餓死:", int(win["reason"].str.contains("MAX_FORWARD_STEPS").sum()))

summary = (win.groupby("window_deg")
              .agg(可行格數=("feasible", "sum"), 總執行秒數=("runtime_s", "sum"))
              .round(1))
summary.index.name = "搜尋範圍(度)"
summary

格數: 90 | 例外: 0 | 被預算餓死: 0


,可行格數,總執行秒數
搜尋範圍(度),,
5.0,6,2747.3
10.0,6,2610.6
20.0,6,2366.5


`window 5 度 -> 6 格可行`，與原表相同——**基準先過**，接線沒接錯。

而 10 度、20 度也都是 **6 格**。逐格比對才知道是不是同一批格子。


In [9]:
wv = (win.assign(v=win.apply(verdict, axis=1))
         .pivot_table(index=["height_mm", "theta_deg"], columns="window_deg",
                      values="v", aggfunc="first"))
wv.columns = ["win5", "win10", "win20"]
wv["一致?"] = np.where((wv.win5 == wv.win10) & (wv.win10 == wv.win20), "", "<<< 變了")

fc = win.pivot_table(index=["height_mm", "theta_deg"], columns="window_deg",
                     values="frame_count", aggfunc="first")
same_feas = int((win[win.window_deg == 5.0].set_index(["height_mm","theta_deg"])["feasible"]
                 == win[win.window_deg == 20.0].set_index(["height_mm","theta_deg"])["feasible"]).sum())
print(f"可行性 5 度 vs 20 度相同 : {same_feas} / 30")
print(f"frame_count 逐位元相同   : {int((fc[5.0] == fc[20.0]).sum())} / 30")
wv

可行性 5 度 vs 20 度相同 : 30 / 30
frame_count 逐位元相同   : 28 / 30


win5                   win10                   win20     一致?
height_mm theta_deg                                                                                
120.0     40.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          45.0                         FEAS                    FEAS                    FEAS        
          50.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          55.0                         FEAS                    FEAS                    FEAS        
          60.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          65.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          70.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          75.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          80.0       NO_SLIP_REQUIRES_NEGAT  COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO  <<< 變了
          85.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
140.0     40.0                         FEAS                    FEAS                    FEAS        
          45.0                         FEAS                    FEAS                    FEAS        
          50.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          55.0                         FEAS                    FEAS                    FEAS        
          60.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          65.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          70.0                         FEAS                    FEAS                    FEAS        
          75.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          80.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          85.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
160.0     40.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          45.0       NO_SLIP_REQUIRES_NEGAT  COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO  <<< 變了
          50.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          55.0       NO_LEGAL_CORNER_PIVOT_  NO_LEGAL_CORNER_PIVOT_  NO_LEGAL_CORNER_PIVOT_        
          60.0       NO_LEGAL_CORNER_PIVOT_  NO_LEGAL_CORNER_PIVOT_  NO_LEGAL_CORNER_PIVOT_        
          65.0       NO_LEGAL_CORNER_PIVOT_  NO_LEGAL_CORNER_PIVOT_  NO_LEGAL_CORNER_PIVOT_        
          70.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          75.0       NO_LEGAL_CORNER_PIVOT_  NO_LEGAL_CORNER_PIVOT_  NO_LEGAL_CORNER_PIVOT_        
          80.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          85.0       NO_LEGAL_CORNER_PIVOT_  NO_LEGAL_CORNER_PIVOT_  NO_LEGAL_CORNER_PIVOT_

### 5.1 結果：放寬四倍，判定【零改變】

```text
可行性        30/30 格完全相同
frame_count   28/30 格逐位元相同
```

只有兩格「變了」，而且變的只是失敗**原因**，兩格都還是不可行：

```text
h=120 th=80   NO_SLIP_REQUIRES_NEGATIVE_X -> COUPLED_RESET   frames  99 -> 132
h=160 th=45   NO_SLIP_REQUIRES_NEGATIVE_X -> COUPLED_RESET   frames  79 ->  90
```

「多走了幾步，然後撞到同一面牆」。

**而這兩格同時是「參數真的有生效」的證據**——必須排除「根本沒接到」：
它們走得更遠且換了失敗模式，而且三組 window 的總執行時間不同
（2747 / 2611 / 2367 秒），不是同一份快取。


---

## 6. 第三個軸：arc_samples（輪緣取樣密度）

§5 之前一直把這個軸標成「還沒檢定」。現在檢定了。

### 6.1 又一個 count vs quantity 陷阱

`sample_match_tolerance = 3` 是**索引數**，不是角度：

```text
arc_samples=121   每格 0.4435 度  ->  3 索引 = 1.330 度
arc_samples=181   每格 0.2957 度  ->  3 索引 = 0.887 度
arc_samples=241   每格 0.2217 度  ->  3 索引 = 0.665 度
```

**同樣的「3」，在 241 下容許量只剩一半。**
直接調高 `arc_samples` 會**偷偷收緊**角落樞轉的約束，
量到的是「容許量變嚴」而不是「輪緣變細」。

修正：把容許量固定成**角度**（tol 隨 arc 等比 -> 3/4/6 索引）。

> 這是本 notebook 第三次遇到同一族陷阱（步數預算、接縫 guard、本項）。
> **凡是整數上限，先問「它是次數還是量」。**


In [10]:
arc = pd.read_csv(DAY13 / "day13_d1_arc_samples.csv")
print("格數:", len(arc),
      "| 例外:", int((arc["stage"] == "EXCEPTION").sum()),
      "| 被預算餓死:", int(arc["reason"].str.contains("MAX_FORWARD_STEPS").sum()))

(arc.groupby(["arc_samples", "sample_tol"])
    .agg(可行格數=("feasible", "sum"))
    .rename_axis(["輪緣取樣", "容許索引數"]))

格數: 90 | 例外: 0 | 被預算餓死: 0


,,可行格數
輪緣取樣,容許索引數,
121,3,6
181,4,10
241,6,4


`arc 121 -> 6 格`，與原表相同（基準先過）。
但 **6 -> 10 -> 4 是非單調的**，完全沒有收斂的樣子。


In [11]:
av = (arc.assign(v=arc.apply(verdict, axis=1))
         .pivot_table(index=["height_mm", "theta_deg"], columns="arc_samples",
                      values="v", aggfunc="first"))
av.columns = ["arc121", "arc181", "arc241"]
av["一致?"] = np.where((av.arc121 == av.arc181) & (av.arc181 == av.arc241),
                       "", "<<< 變了")
print("判定隨取樣改變:", int((av["一致?"] != "").sum()), "/ 30")

S = {a: set(map(tuple, arc[(arc.arc_samples == a) & arc.feasible]
                [["height_mm", "theta_deg"]].values))
     for a in [121, 181, 241]}
print("某個取樣下可行 :", len(S[121] | S[181] | S[241]), "格")
print("三種取樣都可行 :", sorted(S[121] & S[181] & S[241]))
av

判定隨取樣改變: 22 / 30
某個取樣下可行 : 13 格
三種取樣都可行 : [(np.float64(120.0), np.float64(45.0))]


arc121                  arc181                  arc241     一致?
height_mm theta_deg                                                                                
120.0     40.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          45.0                         FEAS                    FEAS                    FEAS        
          50.0       COUPLED_RESET_COLLISIO                    FEAS  COUPLED_RESET_COLLISIO  <<< 變了
          55.0                         FEAS                    FEAS  COUPLED_RESET_COLLISIO  <<< 變了
          60.0       COUPLED_RESET_COLLISIO                    FEAS  COUPLED_RESET_COLLISIO  <<< 變了
          65.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          70.0       COUPLED_RESET_COLLISIO                    FEAS                    FEAS  <<< 變了
          75.0       COUPLED_RESET_COLLISIO                    FEAS  COUPLED_RESET_COLLISIO  <<< 變了
          80.0       NO_SLIP_REQUIRES_NEGAT  COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO  <<< 變了
          85.0       COUPLED_RESET_COLLISIO                    FEAS  COUPLED_RESET_COLLISIO  <<< 變了
140.0     40.0                         FEAS                    FEAS  COUPLED_RESET_COLLISIO  <<< 變了
          45.0                         FEAS  NO_SLIP_REQUIRES_NEGAT                    FEAS  <<< 變了
          50.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          55.0                         FEAS  COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO  <<< 變了
          60.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          65.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          70.0                         FEAS  COUPLED_RESET_COLLISIO  NO_SLIP_REQUIRES_NEGAT  <<< 變了
          75.0       COUPLED_RESET_COLLISIO                    FEAS  COUPLED_RESET_COLLISIO  <<< 變了
          80.0       COUPLED_RESET_COLLISIO                    FEAS                    FEAS  <<< 變了
          85.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
160.0     40.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO        
          45.0       NO_SLIP_REQUIRES_NEGAT  NO_LEGAL_CORNER_PIVOT_  COUPLED_RESET_COLLISIO  <<< 變了
          50.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO  NO_LEGAL_CORNER_PIVOT_  <<< 變了
          55.0       NO_LEGAL_CORNER_PIVOT_  COUPLED_RESET_COLLISIO  NO_LEGAL_CORNER_PIVOT_  <<< 變了
          60.0       NO_LEGAL_CORNER_PIVOT_  NO_LEGAL_CORNER_PIVOT_  NO_SLIP_REQUIRES_NEGAT  <<< 變了
          65.0       NO_LEGAL_CORNER_PIVOT_  NO_LEGAL_CORNER_PIVOT_  COUPLED_RESET_COLLISIO  <<< 變了
          70.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO  NO_LEGAL_CORNER_PIVOT_  <<< 變了
          75.0       NO_LEGAL_CORNER_PIVOT_  COUPLED_RESET_COLLISIO  NO_LEGAL_CORNER_PIVOT_  <<< 變了
          80.0       COUPLED_RESET_COLLISIO  COUPLED_RESET_COLLISIO  NO_LEGAL_CORNER_PIVOT_  <<< 變了
          85.0       NO_LEGAL_CORNER_PIVOT_  NO_LEGAL_CORNER_PIVOT_  COUPLED_RESET_COLLISIO  <<< 變了

### 6.2 結果：73% 的格子會變，只有【一格】三種取樣都可行

```text
判定改變        22 / 30 格 = 73%
某取樣下可行     13 格
三種都可行        1 格   <- 只有 h=120 th=45
```

三個軸的敏感度排序：

| 軸 | 判定改變 | 為什麼 |
|---|---|---|
| `arc_samples` | **22/30 = 73%** | 接觸點的**身分**隨密度改變 |
| `beta_step` | 5/30 = 17% | 候選解**位置**被步長綁死 |
| `beta_search_window` | 0/30 = 0% | 只是超集，判定不可能翻面 |

補償掉接縫 guard 與樞轉容許量之後**還有 73%**，
所以主因是「**哪一個取樣點是接觸點**」本身隨密度改變——
而整條軌跡是靠「釘住同一個 material sample」推進的。


In [12]:
n160 = (arc[arc.height_mm == 160.0]
        .groupby("arc_samples")
        .agg(可行格數=("feasible", "sum"),
             樞轉失敗=("reason", lambda s: s.str.contains("NO_LEGAL_CORNER").sum())))
n160.index.name = "輪緣取樣"
print("h = 160 mm：")
n160

h = 160 mm：


,可行格數,樞轉失敗
輪緣取樣,,
121,0,5
181,0,4
241,0,5


### 6.3 【唯一穩健的結果】h = 160 mm 三種取樣全部不可行

```text
h=160 可行格數    121 -> 0    181 -> 0    241 -> 0
```

**這是整份分析裡唯一站得住的結論。**

不成立的是另一半：140 mm 的可行格數隨取樣在 4/2/2 之間跳，而且是**不同的格子**。

```text
可以講   滾動在 160 mm 不可行（三種取樣一致）
不能講   滾動在 140 mm 可行（取樣一換就換格子）
所以     天花板【上界】160 mm 成立，【下界】沒有量到
```


---

## 7. 結論

### 7.1 三個軸的最終圖像

```text
真正穩健的     h=160 mm 不可行（arc 三種一致）
判準敏感度     arc_samples 73%  >  beta_step 17%  >  window 0%
機制          候選解【位置】與接觸點【身分】都被離散化綁死
              不是「搜尋不夠努力」—— window 已證明無效
```

### 7.2 論文能寫的最強版本

> 滾動越障在 160 mm 於所有測試的取樣密度下皆不可行；
> 140 mm 以下的可行性判準對輪緣取樣密度高度敏感
> （30 格中 22 格隨密度改變，僅 1 格在三種密度下皆可行），
> 因此本文不宣稱一個夾緊的滾動高度上限。

比「>140 mm、<=160 mm」誠實，而且**每個字都有量測支撐**。

### 7.3 D1 / D2 的建議

```text
不要   掃 142-158 mm      —— 判準在該區間本來就不穩定
不要   靠放寬搜尋範圍修    —— 已實測無效（§5）
D2     20 mm 前提確認成立（是資料缺口），但【先不跑】——
       判準不穩定時填缺口只會得到一張不可信的表
要     要有穩定判準，得讓候選解與接觸點身分【不隨離散化平移】
```

### 7.4 三個軸都做完之後，還沒回答的

```text
本輪三個軸都是【滾動側】的離散化。
擺動側（SWING）的判準是否同樣敏感，尚未檢定。
而 Day 12 的策略選擇是拿 ROLL 與 SWING 相比的 ——
若兩側的敏感度不同，那個比較本身也要重新檢視。
```
